# 01 — First authorization failure

**Objective:** See how a seemingly reasonable tool can reveal another tenant's data. This notebook intentionally demonstrates a vulnerability. We have **not** implemented authentication, verified identity, or an authorization gateway yet.

## Mental model

**Authentication:** who is the caller?

**Authorization:** can that caller read *this* record?

A tool parameter like `tenant_id` is **not** proof that the caller belongs to that tenant. Our first experiment exposes this mistake.

In [4]:
from pathlib import Path
from investigation_agent_security_lab.claims import load_claims, get_claim_insecure

# Locate the repository root, whether VS Code starts in
# the repository root or the notebooks directory.
cwd = Path.cwd()

PROJECT_ROOT = (
    cwd if (cwd / "pyproject.toml").exists()
    else cwd.parent
)

DATA = PROJECT_ROOT / "data" / "synthetic" / "claims.csv"

claims = load_claims(DATA)

print(f"Working directory: {cwd}")
print(f"Loaded {len(claims)} synthetic claims")

Working directory: c:\Users\shchitt\Downloads\Projects\investigation-agent-security-lab\notebooks
Loaded 6 synthetic claims


## Experiment 1 — expected record

Imagine an employee whose home tenant is `northstar`. The tool receives `northstar` as a user-controlled parameter. Run the lookup.

In [7]:
caller_home_tenant = "northstar"  # Simulation only; NOT verified identity
allowed = get_claim_insecure(claims, tenant_id="northstar", claim_id="C-1042")
print(allowed)

{'tenant_id': 'northstar', 'claim_id': 'C-1042', 'customer_id': 'CUS-100', 'status': 'open', 'loss_type': 'water damage', 'amount_usd': '4200'}


## Experiment 2 — change one argument

Keep the same simulated employee but change the requested tenant to `harbor`. Both tenants have a claim named `C-1042`. Predict the result **before** running the cell.

In [8]:
exposed = get_claim_insecure(claims, tenant_id="harbor", claim_id="C-1042")
print(exposed)
print("Cross-tenant record returned:", exposed is not None and exposed["tenant_id"] != caller_home_tenant)

{'tenant_id': 'harbor', 'claim_id': 'C-1042', 'customer_id': 'CUS-900', 'status': 'open', 'loss_type': 'water damage', 'amount_usd': '18500'}
Cross-tenant record returned: True


## Why this matters

The function checks that the record belongs to the **requested** tenant, but not that the caller is **authorized for** that tenant. It also cannot distinguish two employees with different resource-level grants inside the same tenant.

**Not fixed yet:** We will later pass a trusted, verified principal via server-owned context and make a gateway check policy before reading or disclosing records. Never treat `caller_home_tenant` in this notebook as authenticated.


1. Which specific value is attacker-controlled?
2. Where is the missing authorization check?
3. Would checking `tenant_id == caller_home_tenant` inside the tool be sufficient for all enterprise access rules? Why or why not?

We will implement and test the fix in the **next** notebook/slice.

In [13]:
if exposed["tenant_id"] != caller_home_tenant:
    print("FALSE: Unauthorized access to claim from a different tenant")
else:
    # Proceed with authorized access
    pass

FALSE: Unauthorized access to claim from a different tenant
